# Intelligent Prompt Optimization Assistant: 1D-CNN Model Training
### Major Project — Phase 3
**Objective:** Implement, train, evaluate, and serialize a 1D Convolutional Neural Network (1D-CNN) for prompt quality classification (`Poor`, `Average`, `Good`).

**Deep Learning Architecture Highlights:**
- **Input:** Token sequence ($N=120$ tokens)
- **Embedding Layer:** 128-dimensional dense representations
- **Conv1D Layer 1:** 128 filters, kernel size 5 (captures 5-gram semantic phrases)
- **MaxPooling1D Layer:** Downsampling factor 2
- **Conv1D Layer 2:** 64 filters, kernel size 3 (hierarchical n-gram synthesis)
- **GlobalMaxPooling1D Layer:** Feature extraction across the sequence
- **Dense Layer:** 64 hidden units with ReLU activation
- **Dropout Layer:** Regularization ($p=0.4$)
- **Output Layer:** 3 units with Softmax activation

In [1]:
# Step 1: Install Dependencies (Run if executing on Google Colab)
# !pip install tensorflow keras numpy pandas matplotlib scikit-learn

import os
import sys
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available     : {tf.config.list_physical_devices('GPU')}")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Step 2: Load Preprocessed Numerical Splits
# When in Google Colab, upload 'dataset/preprocessed_splits.json' using files.upload() or Google Drive
SPLITS_PATH = "../dataset/preprocessed_splits.json"
if not os.path.exists(SPLITS_PATH):
    SPLITS_PATH = "preprocessed_splits.json" # In current Colab working dir

with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

meta = data["metadata"]
vocab_size = meta["vocab_size"]
max_len = meta["max_len"]
classes = meta["classes"]
num_classes = meta["num_classes"]

X_train = np.array(data["train"]["X"], dtype=np.int32)
y_train = np.array(data["train"]["y_one_hot"], dtype=np.float32)

X_val = np.array(data["val"]["X"], dtype=np.int32)
y_val = np.array(data["val"]["y_one_hot"], dtype=np.float32)

X_test = np.array(data["test"]["X"], dtype=np.int32)
y_test = np.array(data["test"]["y_one_hot"], dtype=np.float32)
y_test_idx = np.array(data["test"]["y"], dtype=np.int32)

print(f"Train Data: X={X_train.shape}, y={y_train.shape}")
print(f"Val Data  : X={X_val.shape}, y={y_val.shape}")
print(f"Test Data : X={X_test.shape}, y={y_test.shape}")
print(f"Classes   : {classes} (Vocab: {vocab_size})")

In [ ]:
# Step 3: Build the 1D-CNN Architecture
def build_cnn(vocab_sz, emb_dim=128, seq_len=120, n_classes=3, drop_rate=0.4):
    inputs = layers.Input(shape=(seq_len,), name="token_input", dtype="int32")
    x = layers.Embedding(vocab_sz, emb_dim, input_length=seq_len, name="embedding_layer")(inputs)
    x = layers.Conv1D(128, kernel_size=5, padding="same", activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="conv1d_1")(x)
    x = layers.MaxPooling1D(pool_size=2, name="maxpool1d_1")(x)
    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="conv1d_2")(x)
    x = layers.GlobalMaxPooling1D(name="global_max_pool")(x)
    x = layers.Dense(64, activation="relu", name="dense_hidden")(x)
    x = layers.Dropout(drop_rate, name="dropout_layer")(x)
    outputs = layers.Dense(n_classes, activation="softmax", name="quality_output")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Prompt_Quality_1D_CNN")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_cnn(vocab_size, max_len=max_len, n_classes=num_classes)
model.summary()

In [ ]:
# Step 4: Train 1D-CNN with EarlyStopping and Checkpointing
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath="cnn_model.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Step 5: Visualize Training Curves
epochs_range = range(1, len(history.history["loss"]) + 1)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["loss"], "b-o", label="Train Loss")
plt.plot(epochs_range, history.history["val_loss"], "r-o", label="Val Loss")
plt.title("1D-CNN Training & Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["accuracy"], "b-o", label="Train Accuracy")
plt.plot(epochs_range, history.history["val_accuracy"], "r-o", label="Val Accuracy")
plt.title("1D-CNN Training & Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Test Set Evaluation & Metrics Calculation
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"[+] Test Loss    : {test_loss:.4f}")
print(f"[+] Test Accuracy: {test_acc * 100:.2f}%")

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Confusion Matrix
cm = [[0 for _ in range(num_classes)] for _ in range(num_classes)]
for true_lbl, pred_lbl in zip(y_test_idx, y_pred):
    cm[true_lbl][pred_lbl] += 1

print("\nConfusion Matrix (Rows: Actual, Cols: Predicted):")
print(f"{'':<10} | {'Pred Poor':<12} | {'Pred Avg':<12} | {'Pred Good':<12}")
print("-" * 50)
for i, c in enumerate(classes):
    print(f"{c:<10} | {cm[i][0]:<12} | {cm[i][1]:<12} | {cm[i][2]:<12}")

In [ ]:
# Step 7: Download Trained Model (For Google Colab users)
# from google.colab import files
# files.download('cnn_model.keras')
print("[+] Model saved to 'cnn_model.keras'. Download and place in project 'models/' folder.")